# Choose resolution separately along each branch

**Some branches support fine labels; others should stay broad.** We use a fixed three-level synthetic taxonomy and toy coordinates. Spatial position is a separate diagnostic view, not part of the molecular selection objective. This is not anatomical validation.

New to ITCA? [Start with the concept](00_concepts.ipynb).

In [ ]:
from IPython.display import display
import numpy as np
import scanpy as sc
import seaborn as sns
import gauge_sc as gauge

adata = gauge.datasets.audit_example()
group = adata.obs["group"].astype(str)
adata.obs["class"] = np.where(group.str.startswith("Q"), "Q", "M")
adata.obs["subclass"] = group.map({
    "M0": "M01", "M1": "M01", "M2": "M23", "M3": "M23",
    "Q0": "Q0only", "Q1": "Q12", "Q2": "Q12",
})
adata.obs["leaf"] = group
centers = {"M0": [-2, 1], "M1": [-2, -1], "M2": [0, 1], "M3": [0, -1],
           "Q0": [2, 1], "Q1": [2, 0], "Q2": [2, -1]}
adata.obsm["spatial"] = np.array([centers[g] for g in group]) + np.random.default_rng(0).normal(
    0, 0.2, size=(adata.n_obs, 2)
)

## Evaluate child labels within each parent

Proportions are local to each parent. Top-level classes stay fixed. Splits lacking two children or enough observations are retained as ineligible and block deeper splitting.

In [ ]:
evaluation = gauge.evaluate(
    adata, hierarchy=["class", "subclass", "leaf"],
    views={"molecular": {"obsm": "X_pca"}, "spatial": {"obsm": "spatial"}},
    aggregation="fold_mean",
)
selection = gauge.select(evaluation, view="molecular", delta=[0, 0.2, 0.4, 0.8])
display(selection.table.round(3))


## Inspect one global pruning

Dynamic programming weighs each split together with its descendants. A descendant cannot survive a rejected ancestor. The reported objective sums conditional split gains; it is **not flat-partition ITCA+**. Here `delta=0.4` retains fine M labels and the broad Q class.

In [ ]:
selection.write_to_adata(adata, delta=0.4)
palette = dict(zip(adata.obs["gauge_sc_selection"].cat.categories, sc.pl.palettes.vega_20_scanpy))
gauge.pl.hierarchy(selection, delta=0.4, palette=palette)
with sns.plotting_context("paper"):
    sc.pl.embedding(adata, basis="spatial", color="gauge_sc_selection", palette=palette,
                    frameon=False, title="One molecular pruning, shown in toy space")

Tree and map colors identify the same retained terminals. Coordinates never select these labels. Select globally before displaying a section or region; do not silently reselect each crop.

For real data, supply nested annotation columns and a prepared target molecular view. Transfer is external. [Hierarchy contract and diagnostics](../docs/api.md#hierarchy-evaluation-and-selection) · [Direct penalties versus population ratios](../docs/recipes.md#hierarchy-and-spatial-selection).

## Work with your agent

[Install the skill and choose an interface](../docs/agents.md). Then try:

> Use $gauge to select branch-specific granularity in this hierarchy using molecular support. Recommend a penalty path, show where different branches stop, and explain conditional split gains. Use spatial coordinates for the contextual readout unless I explicitly request spatial selection.

These tutorial settings are an example; adapt them to your question and data.